# TP 3 — Stabiliser puis transformer chaque token

LayerNorm → feed-forward `C → 4C → C` → connexion résiduelle. Ces opérations **ne mélangent pas les positions** : elles agissent sur chaque vecteur indépendamment.

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## Étape 1 — Coder LayerNorm

In [2]:
class LayerNorm1d:
    def __init__(self, dim, eps=1e-5):
        self.eps = eps
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self, x):
        xmean = x.mean(dim=-1, keepdim=True)                 # moyenne par token
        xvar = x.var(dim=-1, keepdim=True, unbiased=False)    # variance par token
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        return self.gamma * xhat + self.beta

    def parameters(self):
        return [self.gamma, self.beta]

### Réponses
- **Axe** : le **dernier** (`dim=-1`), celui des features. On normalise chaque token à travers ses C composantes.
- **ε** : évite une division par zéro (ou par un nombre minuscule) si toutes les composantes d'un vecteur sont égales.
- **γ et β** : paramètres appris (init 1 et 0). La normalisation force moyenne 0 / écart-type 1, ce qui peut être trop contraignant ; γ (échelle) et β (décalage) rendent au réseau la liberté de choisir la distribution utile — il peut même annuler la normalisation si c'est optimal.

## Étape 2 — Choisir l'axe de normalisation

In [3]:
torch.manual_seed(1337)
module = LayerNorm1d(100)

x = torch.randn(32, 100)
x = module(x)

print("x[:, 0] (une feature, 32 tokens)  :", x[:, 0].mean().item(), x[:, 0].std().item())
print("x[0, :] (un token, 100 features)  :", x[0, :].mean().item(), x[0, :].std().item())

x[:, 0] (une feature, 32 tokens)  : 0.14759676158428192 0.8847486972808838
x[0, :] (un token, 100 features)  : -2.2649764730431343e-08 1.005033254623413


In [4]:
x = torch.randn(32, 100)
print(torch.allclose(module(x), nn.LayerNorm(100)(x), atol=1e-6))

True


### Réponse
Prédiction : c'est **`x[0, :]`** qui donne ≈ (0, 1), puisque la normalisation porte sur les features d'un token. `x[:, 0]` n'est pas contraint (ce serait le comportement d'une **BatchNorm**). L'écart-type affiché est ≈ 1,005 et non exactement 1 car `.std()` est non biaisé (division par n−1) alors que la normalisation utilise n.

✔ Identique à `nn.LayerNorm` (dont γ=1 et β=0 à l'init).

## Étape 3 — Construire le réseau feed-forward

In [5]:
class FeedForward(nn.Module):
    def __init__(self, n_embd, mult=4, act=nn.ReLU):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, mult * n_embd),
            act(),
            nn.Linear(mult * n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)

torch.manual_seed(1337)
ffwd = FeedForward(100)
xin = torch.randn(32, 100)
delta = ffwd(xin)
print(delta.shape)

hidden = ffwd.net[1](ffwd.net[0](xin))   # activations après ReLU, (32, 400)
print("proportion d'activations > 0 :", (hidden > 0).float().mean().item())

torch.Size([32, 100])
proportion d'activations > 0 : 0.5074218511581421


### Réponses
- **Proportion positive ≈ 50 %** : à l'init, les pré-activations sont centrées autour de 0 et symétriques ; ReLU met à zéro environ la moitié des neurones.
- **Revenir à C** : la sortie est **ajoutée** au flux résiduel `x` de dimension C ; les deux doivent avoir la même forme. Ça permet aussi d'empiler les blocs indéfiniment. La dimension 4C intermédiaire donne de la capacité (plus de neurones, donc plus de « motifs » détectables) sans changer l'interface.

## Étapes 4 et 5 — Ajouter le résiduel (pré-norm)

In [6]:
B, T, n_embd = 4, 8, 64
layer_norm = nn.LayerNorm(n_embd)
ffwd = FeedForward(n_embd)

x = torch.randn(B, T, n_embd)
delta = ffwd(layer_norm(x))
x_out = x + delta
print(x.shape); print(delta.shape); print(x_out.shape)

torch.Size([4, 8, 64])
torch.Size([4, 8, 64])
torch.Size([4, 8, 64])


### Variantes

In [7]:
count = lambda m: sum(p.numel() for p in m.parameters())
C_ = 64
for name, act in [("ReLU", nn.ReLU), ("GELU", nn.GELU)]:
    for mult in (2, 4, 8):
        m = FeedForward(C_, mult, act)
        print(f"{name:4s}  C -> {mult}C -> C : {count(m):>7,} paramètres   (théorie : 2·k·C² + k·C + C = {2*mult*C_**2 + mult*C_ + C_:,})")

# GELU vs ReLU sur les activations
torch.manual_seed(0)
z = torch.randn(10000)
print("ReLU > 0 :", (F.relu(z) > 0).float().mean().item(), "| GELU < 0 :", (F.gelu(z) < 0).float().mean().item())

ReLU  C -> 2C -> C :  16,576 paramètres   (théorie : 2·k·C² + k·C + C = 16,576)
ReLU  C -> 4C -> C :  33,088 paramètres   (théorie : 2·k·C² + k·C + C = 33,088)
ReLU  C -> 8C -> C :  66,112 paramètres   (théorie : 2·k·C² + k·C + C = 66,112)
GELU  C -> 2C -> C :  16,576 paramètres   (théorie : 2·k·C² + k·C + C = 16,576)
GELU  C -> 4C -> C :  33,088 paramètres   (théorie : 2·k·C² + k·C + C = 33,088)
GELU  C -> 8C -> C :  66,112 paramètres   (théorie : 2·k·C² + k·C + C = 66,112)
ReLU > 0 : 0.4918000102043152 | GELU < 0 : 0.5081999897956848


In [8]:
# Sans résiduel : les formes sont identiques...
x_no_res = ffwd(layer_norm(x))
print(x_no_res.shape)
# ... mais l'information d'entrée n'est plus transmise directement :
print("corrélation entrée/sortie avec résiduel :", torch.corrcoef(torch.stack([x.flatten(), x_out.flatten()]))[0, 1].item())
print("corrélation entrée/sortie sans résiduel :", torch.corrcoef(torch.stack([x.flatten(), x_no_res.flatten()]))[0, 1].item())

torch.Size([4, 8, 64])
corrélation entrée/sortie avec résiduel : 0.9733670949935913
corrélation entrée/sortie sans résiduel : 0.018730321899056435


### Bilan du TP 3
- **Dimension conservée** : LayerNorm conserve exactement la forme `(B, T, C)` ; elle ne fait que recentrer/réduire chaque vecteur.
- **Le MLP mélange-t-il les tokens ?** **Non.** Les `nn.Linear` agissent sur le dernier axe ; la même transformation est appliquée à chaque position indépendamment. Seule l'attention fait communiquer les tokens.
- **Paramètres** : 2·k·C² + k·C + C → avec C=64 : 2C ≈ 16,6 k, 4C ≈ 33,1 k, 8C ≈ 66,1 k (linéaire en k). Le choix ReLU/GELU ne change pas le nombre de paramètres ; GELU est une version lisse qui laisse passer un peu de négatif (gradient non nul pour z<0).
- **Connexion résiduelle** : le bloc apprend une **correction** Δ ajoutée à x plutôt qu'une transformation complète. Le gradient remonte directement par l'addition (« autoroute » du gradient), ce qui rend les réseaux profonds entraînables. Sans elle, les formes restent les mêmes mais l'entrée est quasiment perdue (corrélation ≈ 0) et l'optimisation de piles profondes devient instable.
- **Pré-norm** (`x + f(LN(x))`) plutôt que post-norm : le chemin résiduel reste « propre » (jamais normalisé), plus stable à l'entraînement.